In [ ]:
from dotenv import load_dotenv
load_dotenv()

import base64
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

def chat(messages, system=None):
    params = {"model": model, "max_tokens": 4096, "messages": messages}
    if system: params["system"] = system
    return client.messages.create(**params)

def load_image_base64(path):
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")

def make_image_message(image_path, text, media_type="image/png"):
    return [{
        "type": "image",
        "source": {"type": "base64", "media_type": media_type, "data": load_image_base64(image_path)},
    }, {"type": "text", "text": text}]

In [ ]:
# Test 1: Simple image description
messages = [{"role": "user", "content": make_image_message(
    "images/prop1.png", "What do you see in this image? Describe briefly."
)}]
response = chat(messages)
print(response.content[0].text)

In [ ]:
# Test 2: Fire risk assessment
fire_prompt = """
Analyze the satellite image:
1. Locate the primary residence.
2. Estimate % of roof covered by branches.
3. Evaluate wildfire vulnerability.
4. Assess defensible space.
5. Rate fire risk 1-4 (1=Low, 4=Severe).
One sentence per item.
"""
messages = [{"role": "user", "content": make_image_message("images/prop1.png", fire_prompt)}]
response = chat(messages)
print(response.content[0].text)

In [ ]:
# Test 3: Compare two properties
img1 = load_image_base64("images/prop1.png")
img2 = load_image_base64("images/prop2.png")
messages = [{"role": "user", "content": [
    {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": img1}},
    {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": img2}},
    {"type": "text", "text": "Compare fire risk of these two properties. 3 sentences max."},
]}]
response = chat(messages)
print(response.content[0].text)